In [17]:
import h5py
from PIL import Image
import numpy as np
import pandas as pd
import os
import sys
import io
import zarr
from numcodecs import VLenUTF8
from tqdm import tqdm
source_path = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
sys.path.append(source_path)

In [18]:
input_filename='icdar_train_df_iam_rimes_patches_20250615_170212.csv' 
#'icdar_test_public_df_20250716_101521.csv'#'icdar_test_private_df_20250716_101521.csv'#'icdar_train_df_patches_20250515_164130.csv'
running = 'new-laptop'
saved = 'new-laptop'
train_df = pd.read_csv(f"{source_path}\\outputs\\preprocessed_data\\{input_filename}")
train_df=file_IO.change_filename_from_to(train_df, fr=saved, to=running)

In [12]:
train_df.columns

Index(['file_name', 'source', 'x', 'y', 'x2', 'y2', 'n_cc', 'black_ratio',
       'index'],
      dtype='object')

In [6]:
len(train_df)

35685

In [19]:
train_df_pages = train_df.groupby('file_name', as_index=False).first()
print(f"Number of unique pages: {len(train_df_pages)}")

Number of unique pages: 7137


In [20]:
image_sizes = train_df_pages['file_name'].apply(lambda fn: Image.open(fn).size)
train_df_pages['width'] = image_sizes.apply(lambda x: x[0])
train_df_pages['height'] = image_sizes.apply(lambda x: x[1])
train_df_pages['image_size'] = image_sizes
train_df_pages[['width','height']].describe()
max_height_idx = train_df_pages['height'].idxmax()
max_height_file = train_df_pages.loc[max_height_idx, 'file_name']
print("File with max height:", max_height_file)
max_width = train_df_pages['width'].max()
max_height = train_df_pages['height'].max()
print("Max width:", max_width)
print("Max height:", max_height)

File with max height: C:\Users\andre\PhD\Datasets\iam offline\forms\formsA-D\a01-000u.png
Max width: 2479
Max height: 3542


In [ ]:
min_width = train_df_pages['width'].min()
min_height = train_df_pages['height'].min()
print("Min width:", min_width)
print("Min height:", min_height)

Min width: 1125
Min height: 372
Min width: 2230.9162200785995
Min height: 1223.6471954269382


In [21]:
zarr_path = f"C:\\Users\\andre\PhD\Datasets\ICDAR 2013 - Gender Identification Competition Dataset\\contrastive.zarr"
save_images_to_zarr_with_padding(train_df_pages, zarr_path, target_shape=(max_height, max_width,3))

100%|██████████| 7137/7137 [12:09<00:00,  9.78it/s] 


In [9]:
hdf5_path = f"C:\\Users\\andre\PhD\Datasets\ICDAR 2013 - Gender Identification Competition Dataset\hdf5_train_writers_2.h5"
save_images_to_hdf5(train_df_pages, hdf5_path, resize=None)
#save_images_variable_length(train_df_pages, hdf5_path)

Saved 1/1128 images
Saved 101/1128 images
Saved 201/1128 images
Saved 301/1128 images
Saved 401/1128 images
Saved 501/1128 images
Saved 601/1128 images
Saved 701/1128 images
Saved 801/1128 images
Saved 901/1128 images
Saved 1001/1128 images
Saved 1101/1128 images
HDF5 file with images saved.


In [ ]:
zarr_path = f"C:\\Users\\andre\PhD\Datasets\ICDAR 2013 - Gender Identification Competition Dataset\\train_writers.zarr"
save_images_to_zarr(train_df_pages, zarr_path, resize=None)

100%|██████████| 1128/1128 [03:34<00:00,  5.25it/s]


# easy access

In [3]:
def reload_modules():
    import importlib
    import utils.data_loading as data_loading
    import utils.visualization as visualization
    import utils.dataframes as dataframes
    import utils.utils_transforms as u_transforms
    import utils.training_utils as training_utils
    import utils.model_utils as model_utils
    import utils.file_IO as file_IO
    
    importlib.reload(file_IO)
    importlib.reload(data_loading)
    importlib.reload(visualization)
    importlib.reload(dataframes)
    importlib.reload(u_transforms)
    importlib.reload(model_utils)
    importlib.reload(training_utils)

    return data_loading, visualization, dataframes, u_transforms, training_utils, model_utils, file_IO
data_loading, visualization, dataframes, u_transforms, training_utils, model_utils, file_IO = reload_modules()

## functions

In [4]:
def save_images_to_hdf5(df, hdf5_path, resize=None):
    """
    df: DataFrame with 'file_name' column
    resize: tuple (width, height) to resize images, or None to keep original size
    """
    with h5py.File(hdf5_path, 'w') as f:
        num_images = len(df)
        
        # Open first image to get shape
        img = Image.open(df.iloc[0]['file_name']).convert('RGB')
        if resize:
            img = img.resize(resize)
        else:
            resize=img.size
        img_np = np.array(img)
        img_shape = img_np.shape  # (H, W, C)
        
        # Create dataset to hold all images
        dset = f.create_dataset(
            'images', shape=(num_images, *img_shape),
            dtype=img_np.dtype,
            compression="gzip",
            chunks=(1, *img_shape)  # one image per chunk
        )
        
        dt = h5py.special_dtype(vlen=str)
        f.create_dataset('filenames', (num_images,), dtype=dt)
        
        for idx, row in df.iterrows():
            img = Image.open(row['file_name']).convert('RGB')
            if resize:
                img = img.resize(resize)
            dset[idx] = np.array(img)
            f['filenames'][idx] = row['file_name']
            
            if idx % 100 == 0:
                print(f"Saved {idx+1}/{num_images} images")
    print("HDF5 file with images saved.")
def save_images_variable_length(df, hdf5_path):
    with h5py.File(hdf5_path, 'w') as f:
        num_images = len(df)
        
        # Alternative: use object dtype
        dset = f.create_dataset('images', (num_images,), dtype=h5py.special_dtype(vlen=np.dtype('uint8')))
        
        dt_str = h5py.special_dtype(vlen=str)
        f.create_dataset('filenames', (num_images,), dtype=dt_str)
        
        for idx, row in df.iterrows():
            img = Image.open(row['file_name']).convert('RGB')
            buf = io.BytesIO()
            img.save(buf, format='PNG')
            # Convert to numpy array of uint8
            img_bytes = np.frombuffer(buf.getvalue(), dtype=np.uint8)
            dset[idx] = img_bytes
            f['filenames'][idx] = row['file_name']
            
            if idx % 100 == 0:
                print(f"Saved {idx+1}/{num_images} images")
def save_images_to_zarr(df, zarr_path, resize=None):
    num_images = len(df)

    # Open first image to get shape
    img = Image.open(df.iloc[0]['file_name']).convert('RGB')
    if resize:
        img = img.resize(resize)
    else:
        resize = img.size
    img_np = np.array(img)
    img_shape = img_np.shape  # (H, W, C)

    # Create Zarr array
    store = zarr.DirectoryStore(zarr_path)
    root = zarr.group(store=store, overwrite=True)
    zarr_array = root.create_dataset(
        'images',
        shape=(num_images, *img_shape),
        chunks=(1, *img_shape),
        dtype=img_np.dtype,
        compressor=zarr.Blosc(cname='zstd', clevel=3, shuffle=1)
    )
    filenames = root.create_dataset('filenames', shape=(num_images,), dtype=object,object_codec=VLenUTF8() ) # required for variable-length strings)

    for idx, row in tqdm(df.iterrows(), total=num_images):
        img = Image.open(row['file_name']).convert('RGB')
        if resize:
            img = img.resize(resize)
        zarr_array[idx] = np.array(img)
        filenames[idx] = row['file_name']
def pad_image_to_shape(img_np, target_shape):
    """Pad an image numpy array to the target shape (H, W, C)."""
    h, w, c = img_np.shape
    #target_h, target_w, target_c = target_shape
    padded = np.zeros(target_shape, dtype=img_np.dtype)
    padded[:h, :w, :c] = img_np
    return padded
def save_images_to_zarr_with_padding(df, zarr_path,target_shape=None):
    # Step 2: Create Zarr array
    num_images = len(df)
    store = zarr.DirectoryStore(zarr_path)
    root = zarr.group(store=store, overwrite=True)
    zarr_array = root.create_dataset(
        'images',
        shape=(num_images, *target_shape),
        chunks=(1, *target_shape),
        dtype='uint8',
        compressor=zarr.Blosc(cname='zstd', clevel=3, shuffle=1)
    )
    filenames = root.create_dataset(
        'filenames',
        shape=(num_images,),
        dtype=object,
        object_codec=VLenUTF8()
    )

    # Step 3: Read, pad, and save images
    for idx, row in tqdm(df.iterrows(), total=num_images):
        img = Image.open(row['file_name']).convert('RGB')
        img_np = np.array(img)
        padded_img = pad_image_to_shape(img_np, target_shape)
        zarr_array[idx] = padded_img
        filenames[idx] = row['file_name']